# Lecture 1 - Tensors 

Everything we do in PyTorch when training neural networks is built on one fundamental data structure: tensors. Every step of the training process, loading data, running the forward pass, computing gradients, and updating weights relies on tensors. This means that all training and validation data must be converted into tensors before we start the training loop.

To understand how a neural network learns, and how to transform our input data into tensors, we need to understand the tensor format. A typical PyTorch tensor for image data (2D) follors the shape:

- (N, C, H, W)

Where

- N = Batch size 
- C = Number of channels 
- H = Height 
- W = Width

If you come from a GIS background, this may look familiar, similar to a multi‑dimensional array, such as a Sentinel‑2 image loaded with NumPy. The resemblance is intentional. However, PyTorch tensors have several unique capabilities, especially when it comes to GPU acceleration and automatic differentiation. These advantages will become clear by the end of this lecture.

In PyTorch, tensor objects are created using torch.Tensor. By the end of this lecture, you should be able to:

- create tensors

- manipulate tensors

- understand how tensors flow through a neural network

This is the foundation on which all of PyTorch is built.

 ---
## Part 1 - Create Tensors directly from data

If you have a NumPy array, and want to convert to a tensor, use the torch.tensor() function. 

In [41]:
import numpy as np
import torch

# This could be a one-channel image
np_data = np.array([[1, 2, 3],
                    [4, 5, 6]])

print("This is the NumPy array:")
print(type(np_data))
print(np_data)
print()

# Convert NumPy array to PyTorch tensor
tens = torch.tensor(np_data)
print("This is the tensor:")
print(type(tens))
print(tens)
print()

print("They look the similar, but are of different types.") 


This is the NumPy array:
<class 'numpy.ndarray'>
[[1 2 3]
 [4 5 6]]

This is the tensor:
<class 'torch.Tensor'>
tensor([[1, 2, 3],
        [4, 5, 6]])

They look the similar, but are of different types.


In [42]:
# Crucial difference is that we now can use GPU
if torch.cuda.is_available():
    tens_gpu = tens.to("cuda")
    print("Tensor moved to GPU:", tens_gpu.device)
else:
    print("No GPU available on this machine.")

Tensor moved to GPU: cuda:0


 ---
### NumPy vs PyTorch  
NumPy prints arrays using spaces (matrix‑style), while PyTorch prints tensors using commas (Python‑list style). This is why they look slightly different even though the values are the same.

Tensors can run on the GPU and support automatic differentiation. In the example above we convert a NumPy array into a tensor and check whether a GPU is available.

 ---
## Part 2 - Creating tensors from a desired shape

In many cases you know the shape of the tensor you need, even if you don’t yet know the values.
This is very common when initializing model weights.

Below we reuse the shape of our NumPy array and create new tensors filled with ones, zeros, or random values.

**Note:**  
These tensors all have the same shape, but different initialization strategies.
This is exactly how neural network weights are created internally.

In [43]:
import numpy as np
import torch

# Input - The shape of our NumPy array
shape = np_data.shape
print(f"Our NumPy array shape: {shape}")

# Create tensors with the same shape
ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.randn(shape)

print("\nTensor filled with ones:")
print(ones)

print("\nTensor filled with zeros:")
print(zeros)

print("\nTensor filled with random values:")
print(random)


Our NumPy array shape: (2, 3)

Tensor filled with ones:
tensor([[1., 1., 1.],
        [1., 1., 1.]])

Tensor filled with zeros:
tensor([[0., 0., 0.],
        [0., 0., 0.]])

Tensor filled with random values:
tensor([[ 0.2204, -0.0489,  0.6225],
        [-0.4230,  0.6675, -1.3428]])


 ---
## Part 3 - Creating tensors by mimicking another tensor

So far, we have only created tensors from NumPy arrays.
But in many real‑world cases, you already have a tensor, and you want to create another tensor with the same shape, dtype, and device.

This is extremely common when:

- initializing weights

- creating buffers

- generating masks

- building intermediate tensors inside a model

PyTorch provides convenient functions for this, such as torch.randn_like, torch.zeros_like, and torch.ones_like.

Why this matters

- You don’t need to manually extract .shape, .dtype, or .device.

- The new tensor automatically matches the template.

- It keeps your code clean and reduces bugs.

- It mirrors how PyTorch internally initializes many model parameters.

In [ ]:
# Let's use the tensor we created earlier as a template
template = tens

# Create a new tensor with the same shape and dtype
# Here we fill it with random values, but change to float values
mimicked_tensor = torch.randn_like(template, dtype=torch.float)

print("Template tensor:")
print(template)

print("\nMimicked tensor (same shape, new random values):")
print(mimicked_tensor)


Template tensor:
tensor([[1, 2, 3],
        [4, 5, 6]])

Mimicked tensor (same shape, new random values):
tensor([[-1.0663,  1.2738,  0.3578],
        [ 1.2913, -1.9697, -1.9610]])


 ---
## Part 4 - What attributes does a tensor contain?

Every tensor in PyTorch carries three essential attributes that you will use constantly when debugging, inspecting data, or verifying model behavior.

These attributes are:

- Shape (.shape)
    - A tuple describing the tensor’s dimensions.
    This is one of the most useful debugging tools you have.

- Datatype (.dtype)
    - The type of numbers stored in the tensor. The default for most operations is torch.float32.

- Device (.device)
    - Indicates where the tensor is stored and will be used, CPU or GPU.



In [ ]:
# Let's use our previously created tensor: tens
print("Shape:", tens.shape)
print("Datatype:", tens.dtype)
print("Device:", tens.device)

# Note:
# The dtype is int64 because the original NumPy array contained integers.
# PyTorch preserves the dtype when converting from NumPy.


Shape: torch.Size([2, 3])
Datatype: torch.int64
Device: cpu


The dtype of this tensor is torch.int64.
This happens because PyTorch preserves the datatype of the NumPy array it was created from.
If the NumPy array contains integers, the resulting tensor will also be an integer tensor.

The default dtype for most PyTorch operations is torch.float32, but that only applies when PyTorch itself creates the tensor, not when converting from NumPy.

Since neural networks are almost always trained using float32, it is important that you know how to convert a tensor to this datatype.
You do this using the .float() function.

In [48]:
tens_float = tens.float()
print("Converted to float32:", tens_float.dtype)


Converted to float32: torch.float32


 ---
## Exercise 3 - Why is the default datatype for tensors float32?

Reflect on the overview where you learned about the different stages of the neural network learning process. Connect this to the default datatype float32 and explain why neural networks cannot use integers for training.

**Hint**: Remember that the loss function is continuous, even when predicting binary labels. Think about how this affects gradient calculations and weight updates.